# Prior Sensitivity and Baseline Specification Experiments

---

This notebook runs a sensitivity analysis on the static Bayesian solar model from `02_bayesian_static_model.ipynb`. Every experiment keeps the same data, the same 46 monthly shock dummies, and the same Fourier seasonality. What changes is either:

1. **The prior on the latent solar coefficient η** — does the posterior depend on prior family choice?
2. **How temperature enters the baseline** — does a more physically grounded temperature representation improve fit or change η?

If η is robust across all prior variations, it is data-driven, not a prior artefact.

## Experiment inventory

| Name | What changes | Scientific question |
|------|-------------|---------------------|
| `v1_baseline` | Reference model | Anchor for all comparisons |
| `v2_strict_solar` | Exponential prior on η | Does a sceptical prior change the estimate? |
| `v3_normal_likelihood` | Normal instead of Student-T | How much do heavy tails matter? |
| `v4_lognormal_solar` | LogNormal(−1, 0.8) on η | REE-anchored log-symmetric prior |
| `v5_heavy_tail_solar` | HalfCauchy on η | Does data constrain η from above? |
| `v6_wide_solar_only` | HalfNormal(5) on η only | Purely uninformed solar prior |
| `v7_piecewise` | HDD/CDD temperature split | Engle et al. (1986); Bessec & Fouquau (2008) |
| `v8_autoregressive` | Polynomial + lag-7 demand | Elamin & Fukushima (2018) — *excluded from main LOO* |
| `v9_thi` | Temperature-Humidity Index | Valor et al. (2001); Mirasgedis et al. (2007) |

**Note on v8 (autoregressive):** including 7-day lagged demand inflates the LOO score by ~2,300 points because of autocorrelation, not because of better solar identification. It is run and saved like all others but excluded from the main LOO leaderboard. It appears in a separate cell to quantify the autocorrelation effect.

In [1]:
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pandas as pd
import numpy as np
import json
import pymc as pm
import matplotlib.pyplot as plt
import arviz as az
import pytensor
import warnings

pytensor.config.cxx = ""
warnings.filterwarnings("ignore")

# ── build_dynamic_model inlined ───────────────────────────────────────────────
# Defined here so the notebook does not depend on the solar_models.py version
# on disk. The authoritative source with full docstrings is solar_models.py.
def build_dynamic_model(df, config):
    shock_cols   = [c for c in df.columns if c.startswith('shock_')]
    shock_matrix = df[shock_cols].values if shock_cols else None

    with pm.Model() as model:
        alpha        = pm.Normal('alpha',        mu=0,  sigma=0.1)
        beta_weekend = pm.Normal('beta_weekend', mu=-1, sigma=0.3)
        sin_m = df[['sin_k1','sin_k2','sin_k3']].values
        cos_m = df[['cos_k1','cos_k2','cos_k3']].values
        gamma = pm.Normal('gamma', mu=0, sigma=0.5, shape=3)
        delta = pm.Normal('delta', mu=0, sigma=0.5, shape=3)
        seasonality = pm.math.sum(gamma * sin_m + delta * cos_m, axis=1)

        if shock_matrix is not None:
            beta_shocks  = pm.Normal('beta_shocks', mu=0, sigma=0.28,
                                     shape=shock_matrix.shape[1])
            shock_effect = pm.math.dot(shock_matrix, beta_shocks)
        else:
            shock_effect = 0.0

        bt = config.get("baseline_type", "polynomial")
        if bt == "polynomial":
            beta_temp    = pm.Normal('beta_temp',    mu=0, sigma=0.5)
            beta_temp_sq = pm.HalfNormal('beta_temp_sq',   sigma=0.5)
            weather_effect = (beta_temp    * df['temp_scaled'].values
                            + beta_temp_sq * df['temp_scaled'].values ** 2)
        elif bt == "piecewise":
            beta_heating = pm.HalfNormal('beta_heating', sigma=0.5)
            beta_cooling = pm.HalfNormal('beta_cooling', sigma=0.5)
            weather_effect = (beta_heating * df['heating_scaled'].values
                            + beta_cooling * df['cooling_scaled'].values)
        elif bt == "autoregressive":
            beta_temp    = pm.Normal('beta_temp',    mu=0, sigma=0.5)
            beta_temp_sq = pm.HalfNormal('beta_temp_sq',   sigma=0.5)
            rho_lag      = pm.Normal('rho_lag',      mu=0.8, sigma=0.2)
            weather_effect = (beta_temp    * df['temp_scaled'].values
                            + beta_temp_sq * df['temp_scaled'].values ** 2
                            + rho_lag      * df['demand_lag_7_scaled'].values)
        elif bt == "thi":
            beta_thi    = pm.Normal('beta_thi',    mu=0, sigma=0.5)
            beta_thi_sq = pm.HalfNormal('beta_thi_sq',   sigma=0.5)
            weather_effect = (beta_thi    * df['thi_scaled'].values
                            + beta_thi_sq * df['thi_scaled'].values ** 2)
        else:
            raise ValueError(f"Unknown baseline_type: '{bt}'")

        baseline = (alpha + weather_effect
                  + beta_weekend * df['is_weekend'].values
                  + seasonality + shock_effect)

        dist = config["latent_solar"]["dist"]
        if dist == "HalfNormal":
            mu_solar = pm.HalfNormal('mu_solar', sigma=config["latent_solar"]["sigma"])
        elif dist == "Exponential":
            mu_solar = pm.Exponential('mu_solar', lam=config["latent_solar"]["lam"])
        elif dist == "LogNormal":
            mu_solar = pm.LogNormal('mu_solar',
                                    mu=config["latent_solar"]["mu"],
                                    sigma=config["latent_solar"]["sigma"])
        elif dist == "HalfCauchy":
            mu_solar = pm.HalfCauchy('mu_solar', beta=config["latent_solar"]["beta"])
        else:
            raise ValueError(f"Unknown latent_solar dist: '{dist}'")

        # proxy_norm: non-negative (selfcons / demand_std, pre-2020 = 0)
        expected_demand = baseline - (mu_solar * df['proxy_norm'].values)

        sigma_err = pm.HalfNormal('sigma_err', sigma=config["likelihood"]["sigma_err"])
        lik = config["likelihood"]["dist"]
        if lik == "StudentT":
            nu = pm.Exponential('nu', 1/29)
            pm.StudentT('obs', nu=nu, mu=expected_demand, sigma=sigma_err,
                        observed=df['demand_scaled'].values)
        elif lik == "Normal":
            pm.Normal('obs', mu=expected_demand, sigma=sigma_err,
                      observed=df['demand_scaled'].values)
        else:
            raise ValueError(f"Unknown likelihood: '{lik}'")
    return model

print(f"NumPy {np.__version__} | PyMC {pm.__version__} | ArviZ {az.__version__}")
print("build_dynamic_model defined inline (proxy_norm, shock dummies, all baselines).")


NumPy 1.26.4 | PyMC 5.25.1 | ArviZ 0.23.4
build_dynamic_model defined inline (proxy_norm, shock dummies, all baselines).


## 2. Data Loading and Feature Engineering

All experiments share the same base dataset and feature engineering. Three design decisions carry over from the static model notebook:

**`proxy_norm` (not z-scored).** The solar proxy is `selfcons_theoretical_mwh / demand_std`, not a z-score. Mean-centring the proxy produces large negative values for 2015–2022 (when capacity was near zero but the mean is pulled up by 2023–2026), creating physically impossible negative hidden solar. Dividing by `demand_std` alone keeps every value ≥ 0. Pre-2020 values are set to exactly 0 because the signal is not identifiable there (H_t ≈ 50 MWh/day vs 665,000 MWh/day mean demand).

**Monthly shock dummies.** 46 monthly dummies (March 2020 – December 2023) absorb the COVID-19 lockdown and 2022 energy-crisis demand drops. Without them, the model attributes pandemic demand destruction to hidden solar, inflating η.

**Heating/cooling degrees and THI.** These are precomputed here for the piecewise and THI baseline experiments. Heating degrees = max(18 − T, 0); cooling degrees = max(T − 22, 0). The 18°C and 22°C thresholds are standard European building-physics benchmarks for heating and cooling onset.

In [3]:
# Load the fully merged and engineered dataset
df = pd.read_csv('../data/merged/merged_dataset.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# Generate 3 pairs of Fourier harmonics for annual seasonality
for k in range(1, 4):
    df[f'sin_k{k}'] = np.sin(2 * np.pi * k * df['day_of_year'] / 365.25)
    df[f'cos_k{k}'] = np.cos(2 * np.pi * k * df['day_of_year'] / 365.25)

# Store standardizations for back-calculation later
demand_mean, demand_std = df['demand_mwh_day'].mean(), df['demand_mwh_day'].std()
df['demand_scaled'] = (df['demand_mwh_day'] - demand_mean) / demand_std
df['temp_scaled']   = (df['temp_mean_c'] - df['temp_mean_c'].mean()) / df['temp_mean_c'].std()

# Standardize features (Z-scores)
df['proxy_norm'] = df['selfcons_theoretical_mwh'] / demand_std
df.loc[df['year'] < 2020, 'proxy_norm'] = 0.0   # pre-2020: H_t ≈ 50 MWh, unidentified
assert (df['proxy_norm'] >= 0).all(), "proxy_norm must be non-negative everywhere"

# Piecewise (Heating and Cooling Degrees)
# Assuming temp_celsius is your raw, unscaled temperature column
df['heating_degrees'] = np.maximum(18 - df['temp_mean_c'], 0)
df['cooling_degrees'] = np.maximum(df['temp_mean_c'] - 22, 0)
df['heating_scaled'] = (df['heating_degrees'] - df['heating_degrees'].mean()) / df['heating_degrees'].std()
df['cooling_scaled'] = (df['cooling_degrees'] - df['cooling_degrees'].mean()) / df['cooling_degrees'].std()

# Autoregressive (Lag 7)
# Shift the scaled demand down by 7 days. 
# Use bfill() to fill the first 7 days so PyMC doesn't crash on NaNs.
df['demand_lag_7_scaled'] = df['demand_scaled'].shift(7).bfill()

# Temperature-Humidity Index (THI)
# Assuming you have a raw 'humidity' column (percentage 0-100)
df['thi'] = df['temp_mean_c'] - 0.55 * (1 - (df['humidity_mean_pct'] / 100)) * (df['temp_mean_c'] - 14.5)
df['thi_scaled'] = (df['thi'] - df['thi'].mean()) / df['thi'].std()

# ── Monthly shock dummies (March 2020 – December 2023) ───────────────────────
# 46 monthly dummies absorb COVID lockdown and 2022 energy-crisis demand drops.
# build_dynamic_model detects them via df.columns.startswith('shock_').
shock_start  = pd.Timestamp('2020-03-01')
shock_end    = pd.Timestamp('2023-12-31')
for period in pd.period_range(shock_start, shock_end, freq='M'):
    col = f'shock_{period.year}_{period.month:02d}'
    df[col] = (df['date'].dt.to_period('M') == period).astype(float)

n_shock = sum(1 for c in df.columns if c.startswith('shock_'))
print(f"Dataset ready: {len(df):,} rows | proxy_norm ✅ | shock dummies: {n_shock}")


Dataset ready: 4,138 rows | proxy_norm ✅ | shock dummies: 46


## 3. Experiment Configuration

The configuration is loaded from `experiments_config.json`. Each entry specifies:
- **`latent_solar`**: prior distribution and hyperparameters for η
- **`likelihood`**: `StudentT` (robust to outliers) or `Normal` (no heavy tails)
- **`baseline_type`**: `polynomial` (standard), `piecewise`, or `thi`

The `solar_models.py` module translates each JSON entry into a fully specified PyMC model. All models share the same intercept, weekend effect, Fourier seasonality, and 46 monthly shock dummies; only the parts specified above vary.

In [4]:
# 1. Load the configuration file and the data
with open('experiments_config.json', 'r') as f:
    experiments = json.load(f)

os.makedirs('models', exist_ok=True)
results_dict = {}

print(f"Loaded {len(experiments)} experiments:")
for i, cfg in enumerate(experiments, 1):
    solar   = cfg['latent_solar']
    s_param = solar.get('sigma', solar.get('lam', solar.get('beta', '?')))
    s_str   = f"{solar['dist']}({s_param})"
    excl    = '  [excl. from main LOO]' if cfg.get('exclude_from_loo') else ''
    print(f"  {i}. {cfg['name']:30s}  solar={s_str:28s}  "
          f"baseline={cfg.get('baseline_type','polynomial'):14s}{excl}")

Loaded 9 experiments:
  1. v1_baseline                     solar=HalfNormal(0.3)               baseline=polynomial    
  2. v2_strict_solar                 solar=Exponential(2.0)              baseline=polynomial    
  3. v3_normal_likelihood            solar=HalfNormal(0.3)               baseline=polynomial    
  4. v4_lognormal_solar              solar=LogNormal(0.8)                baseline=polynomial    
  5. v5_heavy_tail_solar             solar=HalfCauchy(0.5)               baseline=polynomial    
  6. v6_wide_solar_only              solar=HalfNormal(5.0)               baseline=polynomial    
  7. v7_piecewise                    solar=HalfNormal(0.3)               baseline=piecewise     
  8. v8_autoregressive               solar=HalfNormal(0.3)               baseline=autoregressive  [excl. from main LOO]
  9. v9_thi                          solar=HalfNormal(0.3)               baseline=thi           


## 4. MCMC Pipeline

Each experiment runs 4 NUTS chains × (1,000 warmup + 1,000 sampling) = 4,000 posterior draws, with `target_accept=0.95` to reduce divergences. Completed traces are saved as `.nc` files and reloaded on subsequent runs — re-running this cell never re-samples unless you delete the cache files.

The `numpyro` backend is used for speed. All models use `random_seed=42` for reproducibility.

**Cache invalidation:** the traces saved in `models/` were computed with the old `proxy_scaled` (z-scored proxy). They must be deleted and re-run after the `proxy_norm` fix to obtain correct results. Delete the `models/` directory before re-running if you have stale traces.


In [5]:
# 2. Execute the Pipeline
for config in experiments:
    model_name = config["name"]
    file_path = f"models/trace_{model_name}.nc"
    
    print(f"\n--- Processing: {model_name} ---")
    
    if os.path.exists(file_path):
        print(f"Loading existing trace from {file_path}...")
        trace = az.from_netcdf(file_path)
    else:
        print(f"Building model and running MCMC for {model_name}...")
        
        # Build the model using the imported blueprint and current JSON config
        model = build_dynamic_model(df, config)
        
        with model:
            trace = pm.sample(draws=1000, 
                              tune=1000, 
                              chains=4, 
                              target_accept=0.95, 
                              nuts_sampler="numpyro", 
                              return_inferencedata=True,
                              random_seed=42)
            
            print(f"Saving to {file_path}...")
            az.to_netcdf(trace, file_path)
    
    results_dict[model_name] = trace

print("\nPipeline execution complete!")


--- Processing: v1_baseline ---
Building model and running MCMC for v1_baseline...


  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

...

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

Saving to models/trace_v9_thi.nc...

Pipeline execution complete!


## 5. Retroactive Log-Likelihood Computation

LOO cross-validation (`az.compare`) requires pointwise log-likelihood values, which PyMC does not compute by default when sampling. This cell retroactively adds them for any trace that is missing the `log_likelihood` group, saving back to disk atomically (write to a temp file, then `os.replace`) to avoid file-lock issues on macOS.

This is only needed once: after the first run, the saved traces include the log-likelihood and this cell skips them.


In [ ]:
print("Retroactively computing missing log-likelihoods...")

# Loop through your configurations to calculate the missing data
for config in experiments:
    model_name = config["name"]
    trace = results_dict[model_name]
    
    # Check if the log_likelihood group is missing
    if "log_likelihood" not in trace:
        print(f"Computing for {model_name}...")
        
        # 1. Rebuild the specific model blueprint
        model = build_dynamic_model(df, config)
        
        # 2. Tell PyMC to calculate the pointwise log-likelihood
        with model:
            pm.compute_log_likelihood(trace)
            
        # 3. The Unix File Swap (Bypassing the lock)
        file_path = f"models/trace_{model_name}.nc"
        temp_path = f"models/temp_{model_name}.nc"
        
        print(f"   -> Saving safely to bypass file lock...")
        # Save to a brand new temporary file
        trace.to_netcdf(temp_path)
        
        # Atomically replace the old locked file with the new complete one
        os.replace(temp_path, file_path)

print("\nAll log-likelihoods successfully computed! You are ready to run the comparison.")

Retroactively computing missing log-likelihoods...
Computing for v1_baseline...


Output()

## 6. Model Comparison with LOO Cross-Validation

### What LOO measures

Leave-One-Out cross-validation estimates how well each model predicts a held-out observation, averaged over all 4,138 days. The metric is **Expected Log Predictive Density (ELPD)** — higher is better. `az.compare` reports ELPD in deviance scale (lower = better, to match information criteria conventions).

The key column is `elpd_diff`: the difference in ELPD relative to the best model. A difference of less than ~4 points is within the standard error and should be treated as equivalent. A difference of tens of points is substantial.

### Exclusion of the autoregressive experiment

The autoregressive model (`demand_lag_7_scaled`) was removed from this comparison. Including 7-day lagged demand as a predictor inflated its LOO score by ~2,336 ELPD points over all other models not because it identified η better, but because autocorrelation makes next-week demand trivially predictable from last-week demand. This circular predictor tells us nothing about the hidden solar signal.


In [ ]:
# 1. Compare the models using Leave-One-Out (LOO) cross-validation
# Split: main comparison vs autoregressive
main_results = {k: v for k, v in results_dict.items()
                if k != 'v8_autoregressive'}
ar_results   = {k: v for k, v in results_dict.items()
                if k == 'v8_autoregressive'}

# 1. Main LOO leaderboard (v1–v7, v9)
print("Computing LOO for main experiments (v1–v7, v9)...")
comparison_df = az.compare(main_results, ic="loo", scale="deviance")
print("\n=== MAIN MODEL LEADERBOARD (lower ELPD deviance = better) ===")
display(comparison_df[["rank","elpd_loo","p_loo","elpd_diff","weight","warning"]])

fig, ax = plt.subplots(figsize=(11, 5))
az.plot_compare(comparison_df, ax=ax, textsize=11)
ax.set_title("Bayesian Model Comparison — LOO Expected Predictive Accuracy\n"
             "(models within 4 ELPD points are statistically equivalent)",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# 2. Autoregressive model: separate analysis
print("\n=== AUTOREGRESSIVE MODEL (v8) — SEPARATE ANALYSIS ===")
print("Excluded from main leaderboard: lag-7 demand exploits autocorrelation,")
print("not physical drivers. ELPD advantage reflects predictability, not η quality.\n")

if ar_results:
    ar_trace = ar_results['v8_autoregressive']
    s_ar = az.summary(ar_trace, var_names=['mu_solar'], stat_focus='mean', hdi_prob=0.94)
    ar_mu = s_ar.loc['mu_solar', 'mean']
    ar_lo = s_ar.loc['mu_solar', 'hdi_3%']
    ar_hi = s_ar.loc['mu_solar', 'hdi_97%']
    print(f"  v8 mu_solar: {ar_mu:.4f}  94% HDI [{ar_lo:.4f}, {ar_hi:.4f}]")
    # Compare ELPD difference vs best main model
    try:
        all_models = {**main_results, 'v8_autoregressive': ar_results['v8_autoregressive']}
        full_df = az.compare(all_models, ic="loo", scale="deviance")
        best_main_elpd = comparison_df['elpd_loo'].max()
        ar_elpd        = full_df.loc['v8_autoregressive', 'elpd_loo']
        diff           = best_main_elpd - ar_elpd
        print(f"  ELPD advantage over best main model: {diff:.1f} points")
        print(f"  Interpretation: {diff:.0f} ELPD points from autocorrelation alone,")
        print(f"  not from better identification of hidden solar.")
    except Exception as e:
        print(f"  (Full comparison unavailable: {e})")

# 3. mu_solar posterior comparison (all models)
print("\n=== mu_solar POSTERIOR — ALL EXPERIMENTS ===")
print(f"{'Model':30s}  {'mean':>7}  {'sd':>6}  {'94% HDI':>20}  {'R-hat':>6}  {'note':>20}")
print("-" * 95)

mu_rows = []
for name, trace in results_dict.items():
    try:
        s = az.summary(trace, var_names=['mu_solar'], stat_focus='mean', hdi_prob=0.94)
        mean = s.loc['mu_solar', 'mean']
        sd   = s.loc['mu_solar', 'sd']
        lo   = s.loc['mu_solar', 'hdi_3%']
        hi   = s.loc['mu_solar', 'hdi_97%']
        rhat = s.loc['mu_solar', 'r_hat']
        note = '[AR — circular]' if name == 'v8_autoregressive' else ''
        mu_rows.append({'model': name, 'mean': mean, 'sd': sd, 'lo': lo, 'hi': hi})
        print(f"  {name:28s}  {mean:>7.4f}  {sd:>6.4f}  [{lo:.4f}, {hi:.4f}]  "
              f"{rhat:>6.3f}  {note}")
    except Exception as e:
        print(f"  {name:28s}  ERROR: {e}")

# 4. mu_solar stability chart (main models only)
main_rows = [r for r in mu_rows if r['model'] != 'v8_autoregressive']
if main_rows:
    fig, ax = plt.subplots(figsize=(11, 4))
    x     = range(len(main_rows))
    means = [r['mean'] for r in main_rows]
    errs  = [[r['mean']-r['lo'] for r in main_rows],
             [r['hi']-r['mean'] for r in main_rows]]
    names = [r['model'] for r in main_rows]
    ax.bar(x, means, color='steelblue', alpha=0.75, edgecolor='white')
    ax.errorbar(x, means, yerr=errs, fmt='none',
                color='black', capsize=4, linewidth=1.5)
    ax.axhline(0.39, color='crimson', linewidth=2, linestyle='--',
               label='REE-implied η ≈ 0.39 (2025)')
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=35, ha='right', fontsize=9)
    ax.set_ylabel('η posterior mean (94% HDI bars)', fontsize=11)
    ax.set_title('Prior Sensitivity — μ_solar Across Experiments\n'
                 'Stability confirms data-driven identification (v8 excluded)',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.show()

    spread = max(means) - min(means)
    print(f"\nμ_solar range (main experiments): {min(means):.4f} – {max(means):.4f}  "
          f"(spread = {spread:.4f})")
    if spread < 0.05:
        print("Stable across prior choices — solar signal is data-driven.")
    elif spread < 0.15:
        print("Moderate sensitivity — some prior influence present.")
    else:
        print("High sensitivity — prior is dominating the posterior.")

Computing LOO for main experiments (v1–v7, v9)...


KeyError: 'elpd_loo'